In [20]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv(encoding='utf-8')
# 创建模型
model = ChatDeepSeek(model="deepseek-chat")
# 创建六名玩家
agents = []
for i in range(6):
    agent = create_agent(
        model=model,
        tools=[],
        checkpointer=InMemorySaver(),
        system_prompt=f"你是Player{i+1}，正在参加6人狼人杀游戏。游戏规则如下：六人局配置为2狼人、2平民、1预言家、1女巫。白天所有玩家发言并投票放逐一名玩家，夜晚狼人杀人、预言家验人、女巫救/毒人。你需要根据游戏进程分析局势，判断其他玩家身份，并做出有利于你阵营的决策。如果你是狼人，要隐藏身份并引导投票放逐好人；如果你是好人，要找出狼人并说服其他玩家放逐他们。发言时要自然、有说服力，可以适度伪装或诈身份，但不要过于激进以免被怀疑。请记住你之前的发言和其他玩家的发言，保持逻辑一致性。",
        middleware=[
            SummarizationMiddleware(
                model="deepseek-chat",
                max_tokens_before_summary=500  # 超过 500 tokens 触发摘要
            )
        ]
    )
    agents.append(agent)

In [21]:
import random

# 为每个玩家发放身份
roles = ["狼人", "狼人", "平民", "平民", "预言家", "女巫"]
random.shuffle(roles)

for i, agent in enumerate(agents):
    # 每个玩家独立的 thread_id，确保记忆隔离
    config = {"configurable": {"thread_id": f"player_{i+1}"}}
    
    # 正确格式化字符串（使用 f-string）
    role = roles[i]
    content = f"你的身份是【{role}】。请记住这个身份，不要告诉其他玩家。现在请进行自我介绍。"
    
    response = agent.invoke(
        {
            "messages": [{"role": "user", "content": content}]
        },
        config=config  # 必须传入 config
    )
    
    print(f"Player{i+1} ({role}): 身份已发放")

Player1 (平民): 身份已发放
Player2 (狼人): 身份已发放
Player3 (女巫): 身份已发放
Player4 (预言家): 身份已发放
Player5 (平民): 身份已发放
Player6 (狼人): 身份已发放


In [24]:
import re
import json
import random
import asyncio
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, asdict
from enum import Enum

class Phase(Enum):
    NIGHT_WEREWOLF = "狼人行动"
    NIGHT_PROPHET = "预言家验人"
    NIGHT_WITCH = "女巫行动"
    DAY_ANNOUNCE = "公布死讯"
    DAY_SPEAK = "白天发言"
    DAY_VOTE = "投票放逐"
    GAME_OVER = "游戏结束"

@dataclass
class GameState:
    round: int = 0
    phase: Phase = Phase.NIGHT_WEREWOLF
    alive_players: List[str] = None
    dead_players: List[str] = None
    public_log: List[Dict] = None
    night_deaths: List[str] = None
    votes: Dict[str, str] = None
    witch_potions: Dict[str, bool] = None
    
    def __post_init__(self):
        if self.alive_players is None:
            self.alive_players = [f"Player{i+1}" for i in range(6)]
        if self.dead_players is None:
            self.dead_players = []
        if self.public_log is None:
            self.public_log = []
        if self.night_deaths is None:
            self.night_deaths = []
        if self.votes is None:
            self.votes = {}
        if self.witch_potions is None:
            self.witch_potions = {"save": True, "poison": True}


class WerewolfGameController:
    """狼人游戏控制器 - 阶段拆分版"""
    
    def __init__(self, agents: List, roles: List[str]):
        self.agents = {f"Player{i+1}": agent for i, agent in enumerate(agents)}
        self.roles = {f"Player{i+1}": role for i, role in enumerate(roles)}
        self.state = GameState()
        self.private_info = {}
        
        # 身份识别
        self.werewolves = [p for p, r in self.roles.items() if r == "狼人"]
        self.prophet = [p for p, r in self.roles.items() if r == "预言家"][0] if any(r == "预言家" for r in roles) else None
        self.witch = [p for p, r in self.roles.items() if r == "女巫"][0] if any(r == "女巫" for r in roles) else None
        
        # 夜晚行动缓存
        self.night_actions = {
            "werewolf_target": None,
            "prophet_check": None,
            "witch_save": False,
            "witch_poison": None
        }
    
    def get_config(self, player_id: str) -> Dict:
        return {"configurable": {"thread_id": f"werewolf_game_{player_id}"}}
    
    def _extract_content(self, response: Any) -> str:
        """提取响应文本"""
        if isinstance(response, str):
            return response
        if isinstance(response, dict):
            if "messages" in response:
                last_msg = response["messages"][-1]
                if isinstance(last_msg, dict):
                    return last_msg.get("content", str(last_msg))
                return str(last_msg)
            return response.get("content", str(response))
        if hasattr(response, "content"):
            return response.content
        return str(response)
    
    # ==================== 阶段1: 狼人讨论 ====================
    
    async def phase_werewolf_discussion(self) -> Tuple[Optional[str], List[Dict]]:
        """
        阶段1: 狼人夜间讨论并选择击杀目标
        返回: (击杀目标, 讨论日志)
        """
        print(f"\n{'='*50}")
        print(f"🐺 阶段1: 狼人讨论 (第{self.state.round}夜)")
        print(f"{'='*50}")
        
        alive_wolves = [w for w in self.werewolves if w in self.state.alive_players]
        if not alive_wolves:
            print("  无存活狼人，跳过")
            return None, []
        
        possible_targets = [p for p in self.state.alive_players if self.roles[p] != "狼人"]
        if not possible_targets:
            return None, []
        
        discussion_log = []
        
        # 独狼情况
        if len(alive_wolves) == 1:
            target = await self._werewolf_single_decide(alive_wolves[0], possible_targets)
            self.night_actions["werewolf_target"] = target
            return target, [{"wolf": alive_wolves[0], "target": target, "type": "single_wolf"}]
        
        # 多狼讨论
        print(f"  存活狼人: {', '.join(alive_wolves)}")
        proposals = {}
        
        # 第一轮: 各自提议
        for wolf in alive_wolves:
            target, reason = await self._werewolf_propose(wolf, alive_wolves, possible_targets)
            proposals[wolf] = {"target": target, "reason": reason}
            discussion_log.append({"wolf": wolf, "proposal": target, "reason": reason})
            print(f"    {wolf} 提议杀: {target}")
        
        # 协商统一
        unique_targets = list(set(p["target"] for p in proposals.values() if p["target"]))
        if len(unique_targets) > 1:
            print("意见分歧，进入协商...")
            final_target = await self._werewolf_negotiate(alive_wolves, proposals, possible_targets)
        else:
            final_target = unique_targets[0] if unique_targets else random.choice(possible_targets)
        
        # 通知所有狼人最终结果
        for wolf in alive_wolves:
            config = self.get_config(wolf)
            self.agents[wolf].invoke({
                "messages": [{
                    "role": "user",
                    "content": f"【系统消息】今晚统一击杀 {final_target}，请准备明天的发言策略。"
                }]
            }, config=config)
        
        self.night_actions["werewolf_target"] = final_target
        print(f"  🎯 最终目标: {final_target}")
        
        return final_target, discussion_log
    
    async def _werewolf_single_decide(self, wolf: str, targets: List[str]) -> Optional[str]:
        """独狼决策"""
        config = self.get_config(wolf)
        
        # 优先杀神职
        priority = [t for t in targets if self.roles[t] in ["预言家", "女巫"]]
        target_list = priority if priority else targets
        
        prompt = {
            "messages": [{
                "role": "user",
                "content": f"你是唯一存活的狼人。可选目标: {', '.join(target_list)}\n"
                          f"请回复要击杀的目标（如Player1）："
            }]
        }
        
        response = self.agents[wolf].invoke(prompt, config=config)
        content = self._extract_content(response)
        
        # 解析
        match = re.search(r"Player\d+", content, re.IGNORECASE)
        if match:
            target = match.group(0).capitalize()
            if target in targets:
                return target
        
        return random.choice(target_list) if target_list else None
    
    async def _werewolf_propose(self, wolf: str, all_wolves: List[str], targets: List[str]) -> Tuple[str, str]:
        """狼人提出击杀建议"""
        config = self.get_config(wolf)
        teammates = [w for w in all_wolves if w != wolf]
        
        prompt = {
            "messages": [{
                "role": "user",
                "content": f"你是{wolf}，狼人队友: {', '.join(teammates)}\n"
                          f"可选击杀目标: {', '.join(targets)}\n"
                          f"请分析并提议一个目标，格式:\n"
                          f"提议击杀: PlayerX\n理由: ..."
            }]
        }
        
        response = self.agents[wolf].invoke(prompt, config=config)
        content = self._extract_content(response)
        
        # 解析目标
        match = re.search(r"提议击杀[：:]\s*(Player\d+)", content, re.IGNORECASE)
        target = match.group(1).capitalize() if match else random.choice(targets)
        
        return target, content[:150]
    
    async def _werewolf_negotiate(self, wolves: List[str], proposals: Dict, targets: List[str]) -> str:
        """狼人协商统一目标"""
        # 统计票数
        from collections import Counter
        votes = Counter([p["target"] for p in proposals.values() if p["target"]])
        
        # 告知每个人其他人的选择，让他们确认
        final_votes = {}
        for wolf in wolves:
            others = {k: v for k, v in proposals.items() if k != wolf}
            config = self.get_config(wolf)
            
            prompt = {
                "messages": [{
                    "role": "user",
                    "content": f"队友提议: {others}\n"
                              f"请回复最终决定的统一目标（必须选一个）: PlayerX"
                }]
            }
            
            response = self.agents[wolf].invoke(prompt, config=config)
            content = self._extract_content(response)
            
            match = re.search(r"Player\d+", content, re.IGNORECASE)
            if match:
                final_votes[wolf] = match.group(0).capitalize()
        
        # 取多数
        if final_votes:
            final_count = Counter(final_votes.values())
            return final_count.most_common(1)[0][0]
        
        return random.choice(targets)
    
    # ==================== 阶段2: 预言家验人 ====================
    
    async def phase_prophet_check(self) -> Optional[Dict]:
        """
        阶段2: 预言家验人
        返回: {"target": str, "is_wolf": bool, "real_role": str} 或 None
        """
        print(f"\n{'='*50}")
        print(f"🔮 阶段2: 预言家验人 (第{self.state.round}夜)")
        print(f"{'='*50}")
        
        if not self.prophet or self.prophet not in self.state.alive_players:
            print("  预言家已死亡或不存在，跳过")
            return None
        
        config = self.get_config(self.prophet)
        valid_targets = [p for p in self.state.alive_players if p != self.prophet]
        
        # 历史记录
        history = self.private_info.get(self.prophet, {}).get("checks", [])
        
        prompt = {
            "messages": [{
                "role": "user",
                "content": f"你是预言家。可选查验目标: {', '.join(valid_targets)}\n"
                          f"历史查验: {history}\n"
                          f"请回复: 查验 PlayerX"
            }]
        }
        
        response = self.agents[self.prophet].invoke(prompt, config=config)
        content = self._extract_content(response)
        
        # 解析目标
        match = re.search(r"查验[：:\s]*(Player\d+)", content, re.IGNORECASE)
        target = match.group(1).capitalize() if match else random.choice(valid_targets)
        
        # 获取结果
        real_role = self.roles[target]
        is_wolf = real_role == "狼人"
        
        # 告知结果
        result_msg = f"【系统消息】查验结果: {target} 是 {'🐺 狼人' if is_wolf else '👤 好人'}({real_role})"
        self.agents[self.prophet].invoke({
            "messages": [{"role": "user", "content": result_msg}]
        }, config=config)
        
        # 记录
        if self.prophet not in self.private_info:
            self.private_info[self.prophet] = {"checks": []}
        self.private_info[self.prophet]["checks"].append({
            "round": self.state.round,
            "target": target,
            "result": "狼人" if is_wolf else "好人"
        })
        
        result = {"target": target, "is_wolf": is_wolf, "real_role": real_role}
        self.night_actions["prophet_check"] = result
        
        print(f"  查验 {target} -> {'狼人' if is_wolf else '好人'}")
        return result
    
    # ==================== 阶段3: 女巫行动 ====================
    
    async def phase_witch_action(self) -> Dict:
        """
        阶段3: 女巫救人/毒人
        返回: {"save": bool, "poison_target": Optional[str]}
        """
        print(f"\n{'='*50}")
        print(f"🧙‍♀️ 阶段3: 女巫行动 (第{self.state.round}夜)")
        print(f"{'='*50}")
        
        if not self.witch or self.witch not in self.state.alive_players:
            print("  女巫已死亡或不存在，跳过")
            return {"save": False, "poison_target": None}
        
        config = self.get_config(self.witch)
        potions = self.state.witch_potions
        werewolf_target = self.night_actions.get("werewolf_target")
        
        # 构建提示
        parts = []
        if werewolf_target and potions["save"]:
            parts.append(f"今晚 {werewolf_target} 被狼人击杀。")
            parts.append("你有解药，是否使用？")
        elif not potions["save"]:
            parts.append("你的解药已用完。")
        
        if potions["poison"]:
            alive_others = [p for p in self.state.alive_players if p != self.witch]
            parts.append(f"你有毒药，可毒死一人: {', '.join(alive_others)}")
        else:
            parts.append("你的毒药已用完。")
        
        parts.append("\n请回复格式:\n救: 是/否\n毒: PlayerX/无")
        
        prompt = {
            "messages": [{"role": "user", "content": "\n".join(parts)}]
        }
        
        response = self.agents[self.witch].invoke(prompt, config=config)
        content = self._extract_content(response)
        
        # 解析
        save = False
        poison = None
        
        # 救人
        if potions["save"] and werewolf_target:
            if re.search(r"救[：:\s]*是|使用解药", content) and not re.search(r"不救|不用", content):
                save = True
                self.state.witch_potions["save"] = False
                print(f"  💚 救活 {werewolf_target}")
        
        # 毒人
        if potions["poison"]:
            match = re.search(r"毒[：:\s]*(Player\d+)", content, re.IGNORECASE)
            if match:
                target = match.group(1).capitalize()
                if target in self.state.alive_players and target != self.witch:
                    poison = target
                    self.state.witch_potions["poison"] = False
                    print(f"  ☠️ 毒死 {target}")
        
        result = {"save": save, "poison_target": poison}
        self.night_actions["witch_action"] = result
        return result
    
    # ==================== 阶段4: 公布死讯 ====================
    
    def phase_announce_deaths(self) -> List[str]:
        """
        阶段4: 公布夜晚死亡结果
        返回: 死亡玩家列表
        """
        print(f"\n{'='*50}")
        print(f"☀️ 阶段4: 公布死讯 (第{self.state.round}天)")
        print(f"{'='*50}")
        
        deaths = []
        
        # 狼人击杀
        kill_target = self.night_actions.get("werewolf_target")
        witch_save = self.night_actions.get("witch_action", {}).get("save", False)
        
        if kill_target and not witch_save:
            deaths.append(kill_target)
        
        # 女巫毒杀
        poison_target = self.night_actions.get("witch_action", {}).get("poison_target")
        if poison_target:
            deaths.append(poison_target)
        
        # 更新状态
        self.state.night_deaths = list(set(deaths))
        for d in self.state.night_deaths:
            if d in self.state.alive_players:
                self.state.alive_players.remove(d)
                self.state.dead_players.append(d)
        
        # 记录
        self.state.public_log.append({
            "round": self.state.round,
            "phase": "night_result",
            "deaths": self.state.night_deaths,
            "is_peace": len(self.state.night_deaths) == 0
        })
        
        if self.state.night_deaths:
            print(f"  ☠️ 昨晚死亡: {', '.join(self.state.night_deaths)}")
        else:
            print("  🌙 平安夜")
        
        return self.state.night_deaths
    
    # ==================== 阶段5: 白天发言 ====================
    
    async def phase_day_speak(self) -> Dict[str, str]:
        """
        阶段5: 白天发言
        返回: {player_id: speech_content}
        """
        print(f"\n{'='*50}")
        print(f"🎤 阶段5: 白天发言 (第{self.state.round}天)")
        print(f"{'='*50}")
        
        speeches = {}
        
        for player_id in self.state.alive_players:
            config = self.get_config(player_id)
            
            # 构建上下文
            context = self._build_speak_context(player_id)
            
            prompt = {
                "messages": [{
                    "role": "user",
                    "content": context + "\n请发言分析局势，并暗示你的身份（可以选择跳身份或隐藏）。"
                }]
            }
            
            print(f"\n  {player_id}({self.roles[player_id]}) 发言:")
            response = self.agents[player_id].invoke(prompt, config=config)
            content = self._extract_content(response)
            
            speeches[player_id] = content
            print(f"    {content[:200]}...")
            
            # 记录到公共日志
            self.state.public_log.append({
                "round": self.state.round,
                "phase": "speak",
                "player": player_id,
                "content": content[:100]  # 摘要
            })
        
        return speeches
    
    def _build_speak_context(self, player_id: str) -> str:
        """构建发言上下文"""
        recent = self.state.public_log[-10:]  # 最近10条
        
        context = f"""你是{player_id}，身份{self.roles[player_id]}。
第{self.state.round}天白天，存活玩家: {', '.join(self.state.alive_players)}
已死亡: {', '.join(self.state.dead_players)}
昨晚死亡: {', '.join(self.state.night_deaths) if self.state.night_deaths else '无 (平安夜)'}

近期事件:
"""
        for log in recent:
            context += f"  - {log}\n"
        
        # 添加私有信息（预言家的查验结果）
        if player_id == self.prophet and player_id in self.private_info:
            checks = self.private_info[player_id].get("checks", [])
            if checks:
                context += f"\n你的查验记录: {checks}\n"
        
        return context
    
    # ==================== 阶段6: 投票放逐 ====================
    
    async def phase_vote(self) -> Optional[str]:
        """
        阶段6: 投票放逐
        返回: 被放逐的玩家，或None（平票）
        """
        print(f"\n{'='*50}")
        print(f"🗳️ 阶段6: 投票放逐 (第{self.state.round}天)")
        print(f"{'='*50}")
        
        votes = {}
        
        for player_id in self.state.alive_players:
            config = self.get_config(player_id)
            
            context = self._build_vote_context(player_id)
            prompt = {
                "messages": [{
                    "role": "user",
                    "content": context + "\n请明确投票，格式: '我投票给PlayerX'"
                }]
            }
            
            response = self.agents[player_id].invoke(prompt, config=config)
            content = self._extract_content(response)
            
            # 解析投票
            target = self._parse_vote(content, player_id)
            votes[player_id] = target
            
            print(f"  {player_id}({self.roles[player_id]}) -> {target or '弃票'}")
        
        # 计票
        from collections import Counter
        valid_votes = {k: v for k, v in votes.items() if v}
        
        if not valid_votes:
            print("  无人投票，今天不放逐")
            return None
        
        vote_count = Counter(valid_votes.values())
        print(f"\n  投票统计: {dict(vote_count)}")
        
        max_votes = max(vote_count.values())
        candidates = [p for p, v in vote_count.items() if v == max_votes]
        
        # 需要超过半数
        threshold = len(self.state.alive_players) // 2 + len(self.state.alive_players) % 2
        
        if len(candidates) == 1 and max_votes >= threshold:
            expelled = candidates[0]
            print(f"  ☠️ {expelled} 被放逐！身份: {self.roles[expelled]}")
            
            self.state.alive_players.remove(expelled)
            self.state.dead_players.append(expelled)
            
            self.state.public_log.append({
                "round": self.state.round,
                "phase": "vote",
                "expelled": expelled,
                "role": self.roles[expelled],
                "votes": valid_votes
            })
            
            return expelled
        else:
            print(f"  🤝 平票或不足半数(需{threshold}票)，无人放逐")
            return None
    
    def _build_vote_context(self, player_id: str) -> str:
        """构建投票上下文"""
        return f"""你是{player_id}({self.roles[player_id]})，现在是投票阶段。
存活: {', '.join(self.state.alive_players)}
已死: {', '.join(self.state.dead_players)} ({[self.roles[p] for p in self.state.dead_players]})

根据今天的发言，你必须投票放逐一个最像狼人的玩家。
"""
    
    def _parse_vote(self, content: str, voter: str) -> Optional[str]:
        """解析投票"""
        text = str(content).upper()
        
        # 显式投票
        patterns = [
            r"我投票给[：:\s]*(PLAYER\d+)",
            r"投票[：:\s]*(PLAYER\d+)",
            r"投[给]*[：:\s]*(PLAYER\d+)",
            r"放逐[：:\s]*(PLAYER\d+)",
        ]
        
        for p in patterns:
            match = re.search(p, text)
            if match:
                target = match.group(1).capitalize()
                if target in self.state.alive_players and target != voter:
                    return target
        
        # 弃票检测
        if re.search(r"弃票|不投|过", text):
            return None
        
        return None
    
    # ==================== 辅助功能 ====================
    
    def check_winner(self) -> Optional[str]:
        """检查游戏胜负"""
        alive = self.state.alive_players
        wolves = [p for p in alive if self.roles[p] == "狼人"]
        good = [p for p in alive if self.roles[p] != "狼人"]
        
        if len(wolves) == 0:
            return "good"
        if len(wolves) >= len(good):
            return "werewolf"
        return None
    
    def reset_night_actions(self):
        """重置夜晚行动缓存"""
        self.night_actions = {
            "werewolf_target": None,
            "prophet_check": None,
            "witch_save": False,
            "witch_poison": None
        }
    
    def next_round(self):
        """进入下一回合"""
        self.state.round += 1
        self.reset_night_actions()


# ==================== 使用示例: 完全自定义流程 ====================

async def custom_game_flow():
    """完全自定义的游戏流程示例"""
    
    # 初始化
    controller = WerewolfGameController(agents, roles)
    controller.state.round = 1
    
    print("🎮 游戏开始!")
    print(f"配置: {controller.roles}")
    
    max_rounds = 5
    
    while controller.state.round <= max_rounds:
        r = controller.state.round
        print(f"\n{'#'*60}")
        print(f"# 第 {r} 天")
        print(f"{'#'*60}")
        
        # ========== 夜晚阶段 ==========
        
        # 1. 狼人讨论
        target, wolf_logs = await controller.phase_werewolf_discussion()
        
        # 2. 预言家验人
        check_result = await controller.phase_prophet_check()
        
        # 3. 女巫行动
        witch_result = await controller.phase_witch_action()
        
        # ========== 白天阶段 ==========
        
        # 4. 公布死讯
        deaths = controller.phase_announce_deaths()
        
        # 检查胜负
        winner = controller.check_winner()
        if winner:
            print(f"\n🏆 游戏结束! {'好人' if winner == 'good' else '狼人'}阵营胜利!")
            break
        
        # 5. 白天发言
        speeches = await controller.phase_day_speak()
        
        # 6. 投票放逐
        expelled = await controller.phase_vote()
        
        # 再次检查胜负
        winner = controller.check_winner()
        if winner:
            print(f"\n🏆 游戏结束! {'好人' if winner == 'good' else '狼人'}阵营胜利!")
            break
        
        # 进入下一回合
        controller.next_round()
    
    # 游戏结束 reveal
    print(f"\n{'='*50}")
    print("📜 最终身份:")
    for p, r in controller.roles.items():
        status = "🟢存活" if p in controller.state.alive_players else "🔴死亡"
        print(f"  {p}: {r} ({status})")


# 运行
await custom_game_flow()

🎮 游戏开始!
配置: {'Player1': '平民', 'Player2': '狼人', 'Player3': '女巫', 'Player4': '预言家', 'Player5': '平民', 'Player6': '狼人'}

############################################################
# 第 1 天
############################################################

🐺 阶段1: 狼人讨论 (第1夜)
  存活狼人: Player2, Player6
    Player2 提议杀: Player1
    Player6 提议杀: Player3
意见分歧，进入协商...
  🎯 最终目标: Player3

🔮 阶段2: 预言家验人 (第1夜)
  查验 Player3 -> 好人

🧙‍♀️ 阶段3: 女巫行动 (第1夜)
  ☠️ 毒死 Player2

☀️ 阶段4: 公布死讯 (第1天)
  ☠️ 昨晚死亡: Player2, Player3

🎤 阶段5: 白天发言 (第1天)

  Player1(平民) 发言:
    content='（语气震惊但迅速冷静）双死！昨晚女巫开毒了，而且狼刀也落下了。2号和3号同时倒牌，说明女巫毒了其中一个，狼刀了另一个。现在场上只剩四个人，情况非常危急，大概率是两狼两好，或者一狼三好但女巫没药了。\n\n我这里是张好人牌，平民身份。现在必须尽快理清2号和3号的身份。如果女巫还在场，请跳出来说明毒的是谁，这样我们能确定另一个吃刀的是好人还是狼人。如果女巫不跳，我们只能从之前的发言和...

  Player4(预言家) 发言:
    content='（语气凝重）双死局，现在只剩四个人了。昨晚我验了Player3，他是好人，但被毒了，说明女巫可能判断失误。现在场上很可能只剩一神一民两狼，或者两民两狼。我是预言家，昨天验3号是金水，但女巫毒了他，这很伤。现在我的身份必须跳明了，否则好人没法打。\n\nPlayer5是我第一天验的金水，他是铁好人。所以狼坑就在Player1和Player6里面。今天我们必须从1号和6号里出一个。我...

  Player5(平民) 发言:
    conten